In [ ]:
!wget -q -c -O rev1-benchmark-data.tar.gz https://github.com/huboqiang/omnialigner/releases/download/rev1/rev1-benchmark-data.tar.gz
!git clone --quiet --depth 1 --branch rev1 https://github.com/huboqiang/omnialigner.git omnialigner-rev1

from pathlib import Path
import hashlib, tarfile, sys
assert (3, 10) <= sys.version_info[:2] < (3, 12), "Use a Python 3.10 or 3.11 Jupyter kernel."
archive = Path("rev1-benchmark-data.tar.gz").resolve()
checkout = Path("omnialigner-rev1").resolve()
assert archive.is_file() and (checkout / ".git").is_dir(), "Download or clone failed; inspect the messages above."
assert hashlib.sha256(archive.read_bytes()).hexdigest() == "9b98464f0b6625bae48514c601b740caee50aeb008525f103a362fc7a630d7d3", "Data archive checksum mismatch"
destination = checkout / "benchmarks/rev1"
with tarfile.open(archive, "r:gz") as bundle:
    for member in bundle.getmembers():
        if member.name == "NCS" or member.name.startswith("NCS/"):
            member.name = "OmniAligner" + member.name[3:]
        target = (destination / member.name).resolve()
        assert target.is_relative_to(destination) and not (member.issym() or member.islnk()), "Unsafe archive member"
    bundle.extractall(destination)
PACKAGE_ROOT = destination / "OmniAligner"
print("Downloaded and verified rev1 data; package:", PACKAGE_ROOT.name)


In [ ]:
%pip install -q -r {PACKAGE_ROOT / "requirements.txt"}
%cd {PACKAGE_ROOT}


## Download and run
Use a **fresh Python 3.10/3.11 Jupyter kernel** with `wget` and `git` available. The first two commands download the public data archive and clone the `rev1` branch. No private directory or credentials are required. The six-dataset scoring run is CPU-based; allow 48–64 GB RAM for the largest dataset. The table-only and DLPFC examples are smaller.


# OmniAligner: reproduce the selected NCS results

This notebook recomputes the six reported metrics from **original expression/annotations + selected NPZ coordinates**. No aligned H5AD export is needed. The selected weights are verified and archived alongside the coordinates; scoring does not refit the model or replay the optimiser.

The reference CSV is used only for the final comparison, never to generate recomputed scores. Six datasets, the original adjacent pairs and the original evaluation gene arguments are retained. These are selected development results, with the original annotation-assisted evaluation limitations.

The release data contain exact original expression values for the frozen evaluation-gene panel, original cell IDs, annotations and raw coordinates, exported losslessly as sparse NPZs. They are sufficient for the scoring computation; full H5AD downloads are linked separately. Input hashes bind the exported arrays to the audited source H5ADs. No benchmark-server directory is required.

## 1. Load the downloaded benchmark inputs
The paths below are relative to the cloned package. All 18 MLN pairs are included. Each dataset is saved after scoring; reference values never fill a missing or failed calculation.

In [ ]:
from pathlib import Path
import os, sys, json, csv, hashlib, importlib.metadata, warnings, time
import numpy as np
import pandas as pd
import anndata as ad
import scipy.sparse as sp
from IPython.display import display

def locate_repository():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "notebooks/OmniAligner_NCS_manifest.json").is_file():
            return candidate
    raise FileNotFoundError("Run from this repository or its notebooks directory")
REPO_ROOT = locate_repository()
DATA_ROOT = REPO_ROOT / "data"
INPUT_ROOT = DATA_ROOT
input_records = json.loads((INPUT_ROOT / "evaluation_inputs_manifest.json").read_text())
input_index = {(x["dataset"], x["layer"]): x for x in input_records}
OUTPUT_ROOT = REPO_ROOT / "results" / "omni"
DATASETS = ["DLPFC", "MHPR", "MMPC", "MPC", "E14_16", "MLN"]
VERIFY_SOURCE_HASHES = True
ATOL = 1e-10
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
METRICS = ["PCC", "CosSim", "SSIM", "MI", "MSE", "Region"]
NATIVE_KEYS = ["PCC", "CosSim", "SSIM", "MI", "Expression_MSE_native_ASSD_bygrid", "Accuracy"]
manifest = json.loads((REPO_ROOT / "notebooks/OmniAligner_NCS_manifest.json").read_text())
def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()
def save_json(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False))
    temporary.replace(path)
reference_path = REPO_ROOT / "notebooks/OmniAligner_NCS_reference.csv"
assert sha256(reference_path) == manifest["expected_table_sha256"]
reference = pd.read_csv(reference_path).set_index("Dataset")
print("Reference values (not recomputed yet)")
display(reference[METRICS].round(6))

## 2. Verify inputs and the frozen scorer
Hashes bind the exact source H5ADs, selected coordinates, nine checkpoints and original scoring implementation. The six metric directions are PCC/CosSim/SSIM/MI/Region ↑ and MSE ↓. MSE scales differ across datasets; no cross-dataset mean is calculated.

In [ ]:
sys.path.insert(0, str(REPO_ROOT / "src"))

inventory = []
for dataset in DATASETS:
    config = manifest["datasets"][dataset]
    assert sha256(DATA_ROOT / config["coordinates"]) == config["coordinates_sha256"], dataset
    for item in config["weights"]:
        assert sha256(DATA_ROOT / item["path"]) == item["sha256"], item["path"]
    for layer in range(len(config["sections"])):
        record = input_index[(dataset, layer)]
        source = INPUT_ROOT / record["path"]
        assert source.is_file(), source
        assert record["source_h5ad_sha256"] == config["source_sha256"][layer]
        if VERIFY_SOURCE_HASHES:
            assert sha256(source) == record["sha256"], source
    inventory.append({"Dataset": dataset, "Sections": len(config["sections"]), "Pairs": config["pairs"],
                      "Original cells": config["original_cells"], "Scored cells": config["evaluation_cells"],
                      "Gene arguments": len(config["evaluation_genes"]), "Unique genes": len(set(config["evaluation_genes"]))})
display(pd.DataFrame(inventory).set_index("Dataset"))
save_json(OUTPUT_ROOT / "INPUT_VALIDATION.json", {"source_hashes_checked": VERIFY_SOURCE_HASHES, "datasets": DATASETS, "inventory": inventory})

## Metric calculations, shown explicitly

For every adjacent section pair, `coords` contains the two XY arrays and a shared **10 × 10 overlap grid**. For each gene, `X` contains its two expression vectors. We call the functions below directly: `for metric in [PCC, CosSim, SSIM, MI, MSE]: metric(X, coords)`. `Region(labels, coords)` measures nearest-neighbor regional-label agreement.

| Metric | Calculation |
|---|---|
| PCC ↑ | Pearson correlation between the two vectors of grid-mean expression |
| CosSim ↑ | Dot product divided by the product of vector norms |
| SSIM ↑ | Native luminance × contrast × structure, after separately scaling each vector by its maximum |
| MI ↑ | Native `mutual_info_regression`, with seed 0 reset for each gene |
| MSE ↓ | Mean squared difference between the unscaled grid-mean vectors |
| Region ↑ | Fraction of moving-section labels matching the nearest reference label inside the overlap box |

The grid uses the original half-open bin boundaries and native density threshold. Region uses the original closed overlap bounds. Gene arguments (including repeated names) and adjacent pairs receive equal weights. Missing/nonfinite values are not silently dropped. Native insufficient-grid fallback values remain explicit and are excluded from eligible comparison ranks.

In [ ]:
import math
from scipy.stats import pearsonr
from scipy.spatial import cKDTree
from sklearn.feature_selection import mutual_info_regression

### Shared spatial support

In [ ]:
def shared_grid(reference_xy, moving_xy):
    """Use the original overlap box, half-open bins and native density threshold."""
    xy = (reference_xy, moving_xy)
    x_min = max(np.min(reference_xy[:, 0]), np.min(moving_xy[:, 0]))
    x_max = min(np.max(reference_xy[:, 0]), np.max(moving_xy[:, 0]))
    y_min = max(np.min(reference_xy[:, 1]), np.min(moving_xy[:, 1]))
    y_max = min(np.max(reference_xy[:, 1]), np.max(moving_xy[:, 1]))
    bins = []
    for i in range(10):
        for j in range(10):
            x_start = x_min + i * (x_max - x_min) / 10
            y_start = y_min + j * (y_max - y_min) / 10
            x_end = x_start + (x_max - x_min) / 10
            y_end = y_start + (y_max - y_min) / 10
            bins.append((x_start, x_end, y_start, y_end))

    def inside(points, box):
        x_start, x_end, y_start, y_end = box
        return ((points[:, 0] >= x_start) & (points[:, 0] < x_end)
                & (points[:, 1] >= y_start) & (points[:, 1] < y_end))

    counts = [[np.sum(inside(points, box)) for box in bins] for points in xy]
    threshold = min(np.mean(counts[0]), np.mean(counts[1])) / 2
    labels = [np.full(len(points), None, dtype=object) for points in xy]
    counter = 1
    for index, box in enumerate(bins):
        keep = counts[0][index] >= threshold and counts[1][index] >= threshold
        name = f"Grid_{counter}" if keep else "NonOverlap"
        for points, values in zip(xy, labels):
            values[inside(points, box)] = name
        if keep:
            counter += 1
    return {"xy": xy, "grid_labels": labels}

### Expression means in the retained bins

In [ ]:
def grid_expression(X, coords):
    """Mean expression per retained bin, sorted by the native string grid label."""
    means = []
    for expression, labels in zip(X, coords["grid_labels"]):
        frame = pd.DataFrame({"expression": expression, "grid_label": labels})
        frame = frame[(frame.grid_label != "NonOverlap") & frame.grid_label.notnull()]
        means.append(frame.groupby("grid_label").mean().sort_index()["expression"].to_numpy())
    return means

### 1. Pearson correlation

In [ ]:
def PCC(X, coords):
    a, b = grid_expression(X, coords)
    return 0.0 if a.size < 2 or b.size < 2 else pearsonr(a, b).statistic

### 2. Cosine similarity

In [ ]:
def CosSim(X, coords):
    a, b = grid_expression(X, coords)
    return 0.0 if a.size < 2 or b.size < 2 else a.dot(b) / (np.linalg.norm(a) * np.linalg.norm(b))

### 3. Structural similarity

In [ ]:
def SSIM(X, coords):
    a, b = grid_expression(X, coords)
    if a.size < 2 or b.size < 2:
        return 0.0
    # Native convention: separately divide each gene's grid vector by its maximum.
    a, b = a / a.max(), b / b.max()
    mu_a, mu_b = a.mean(), b.mean()
    sigma_a = np.sqrt(((a - mu_a) ** 2).mean())
    sigma_b = np.sqrt(((b - mu_b) ** 2).mean())
    covariance = ((a - mu_a) * (b - mu_b)).mean()
    C1, C2 = 0.01 ** 2, 0.03 ** 2
    C3 = C2 / 2
    luminance = (2 * mu_a * mu_b + C1) / (mu_a ** 2 + mu_b ** 2 + C1)
    contrast = (2 * sigma_a * sigma_b + C2) / (sigma_a ** 2 + sigma_b ** 2 + C2)
    structure = (covariance + C3) / (sigma_a * sigma_b + C3)
    return luminance * contrast * structure

### 4. Mutual information

In [ ]:
def MI(X, coords):
    a, b = grid_expression(X, coords)
    if a.size < 2 or b.size < 2:
        return 0.0
    # Reset the seed as in the frozen scorer, including its jitter convention.
    np.random.seed(0)
    return mutual_info_regression(a.reshape(-1, 1), b)[0]

### 5. Expression mean squared error

In [ ]:
def MSE(X, coords):
    a, b = grid_expression(X, coords)
    if a.size < 2 or b.size < 2:
        return 100.0  # Native fallback, excluded from eligible table rankings.
    return np.mean(np.square(a - b))

### 6. Regional-label agreement

In [ ]:
def Region(labels, coords):
    """Moving-to-reference nearest-neighbor label agreement in the overlap box."""
    reference_xy, moving_xy = coords["xy"]
    x_min = max(np.min(reference_xy[:, 0]), np.min(moving_xy[:, 0]))
    x_max = min(np.max(reference_xy[:, 0]), np.max(moving_xy[:, 0]))
    y_min = max(np.min(reference_xy[:, 1]), np.min(moving_xy[:, 1]))
    y_max = min(np.max(reference_xy[:, 1]), np.max(moving_xy[:, 1]))
    keep = [((points[:, 0] >= x_min) & (points[:, 0] <= x_max)
             & (points[:, 1] >= y_min) & (points[:, 1] <= y_max))
            for points in (reference_xy, moving_xy)]
    reference_xy, moving_xy = reference_xy[keep[0]], moving_xy[keep[1]]
    reference_labels = np.asarray(labels[0])[keep[0]]
    moving_labels = np.asarray(labels[1])[keep[1]]
    _, neighbors = cKDTree(reference_xy).query(moving_xy)
    if len(neighbors) == 0:
        return 0.0
    matches = sum(reference_labels[target] == moving_labels[index]
                  for index, target in enumerate(neighbors)
                  if target < len(reference_labels) and index < len(moving_labels))
    return matches / len(neighbors)

## 3. Join selected coordinates to original cells
The join uses `s<section index>_<original obs_name>`, with an exact one-to-one identity check. For MPC and E14–16 the saved `historical_cohort` mask reproduces the reported common-cell evaluation; the additional original cells are not silently included. DLPFC preserves 123 gene arguments (117 distinct genes), including the historical multiplicity. All other gene orders are also unchanged.

Coordinates remain in the saved benchmark frame, without rescaling or a physical-micrometre claim. Expression is neither normalised nor otherwise altered here.

In [ ]:
def load_scoring_slice(dataset, layer, coordinates):
    config = manifest["datasets"][dataset]
    record = input_index[(dataset, layer)]
    with np.load(INPUT_ROOT / record["path"], allow_pickle=False) as raw:
        matrix = sp.csr_matrix((raw["data"], raw["indices"], raw["indptr"]), shape=tuple(raw["shape"]))
        source = ad.AnnData(X=matrix)
        source.obs_names = raw["ids"].astype(str)
        source.var_names = raw["genes"].astype(str)
        source.obs[config["label_column"]] = raw["labels"].astype(str)
        source.obsm["spatial"] = raw["raw_xy"]
    try:
        wanted = [f"s{layer}_{value}" for value in source.obs_names]
        selected = coordinates["layers"] == layer
        ids = coordinates["ids"][selected].tolist()
        assert len(set(ids)) == len(ids) == len(wanted)
        assert set(ids) == set(wanted), (dataset, layer, "Cell identity mismatch")
        lookup = {value: index for index, value in enumerate(ids)}
        order = np.asarray([lookup[value] for value in wanted])
        xy = coordinates["nonrigid"][selected][order]
        keep = coordinates["historical_cohort"][selected][order] if "historical_cohort" in coordinates.files else np.ones(source.n_obs, dtype=bool)
        assert np.isfinite(xy).all() and xy.shape == (source.n_obs, 2)
        unique_genes = list(dict.fromkeys(config["evaluation_genes"]))
        data = source[keep, unique_genes].copy()
        data.obsm["spatial"] = xy[keep]
        data.obs["Region"] = data.obs[config["label_column"]].astype(str)
        if not sp.issparse(data.X):
            data.X = sp.csr_matrix(data.X)
        return data
    finally:
        if source.isbacked:
            source.file.close()

## 4. Recompute every adjacent pair
For each gene, call the five molecular metric functions shown above with `X` and `coords`; then calculate regional-label agreement separately. Average genes within each pair and give each adjacent pair equal weight. Non-finite values propagate rather than being silently dropped.


In [ ]:
recomputed_rows = []
started = time.time()
for dataset in DATASETS:
    config = manifest["datasets"][dataset]
    target = OUTPUT_ROOT / dataset
    target.mkdir(exist_ok=True)
    pair_rows, gene_rows = [], []
    with np.load(DATA_ROOT / config["coordinates"], allow_pickle=False) as coordinates:
        assert len(coordinates["ids"]) == config["original_cells"]
        assert set(coordinates["layers"].tolist()) == set(range(len(config["sections"])))
        reference_slice = load_scoring_slice(dataset, 0, coordinates)
        for pair_index in range(config["pairs"]):
            moving_slice = load_scoring_slice(dataset, pair_index + 1, coordinates)
            reference_section, moving_section = reference_slice, moving_slice
            gene_arguments = config["evaluation_genes"]
            coords = shared_grid(reference_section.obsm["spatial"], moving_section.obsm["spatial"])
            gene_records = []
            for gene_name in gene_arguments:
                X = (reference_section.X[:, reference_section.var_names == gene_name].toarray().squeeze(),
                     moving_section.X[:, moving_section.var_names == gene_name].toarray().squeeze())
                gene_result = {"gene": gene_name}
                for metric, key in zip([PCC, CosSim, SSIM, MI, MSE], NATIVE_KEYS[:5]):
                    np.random.seed(0)
                    with warnings.catch_warnings(record=True) as caught:
                        warnings.simplefilter("always")
                        try:
                            value = float(metric(X, coords))
                            gene_result[key] = value if math.isfinite(value) else None
                            gene_result[key + "_error"] = "" if math.isfinite(value) else "native_nonfinite"
                        except Exception as error:
                            gene_result[key] = None
                            gene_result[key + "_error"] = repr(error)
                    gene_result[key + "_warnings"] = " | ".join(str(w.message) for w in caught)
                gene_records.append(gene_result)

            # Equal gene-argument weights. A missing gene score invalidates that pair mean.
            values = {}
            for key in NATIVE_KEYS[:5]:
                numbers = [row[key] for row in gene_records]
                assert len(numbers) == len(gene_arguments)
                values[key] = math.fsum(numbers) / len(numbers) if all(v is not None and math.isfinite(v) for v in numbers) else None

            labels = (reference_section.obs["Region"].to_numpy(), moving_section.obs["Region"].to_numpy())
            values["Accuracy"] = float(Region(labels, coords))
            valid = [pd.notna(g) & (g != "NonOverlap") for g in coords["grid_labels"]]
            valid_grids = [len(set(g[mask])) for g, mask in zip(coords["grid_labels"], valid)]
            support = {"reference_cells": reference_section.n_obs, "moving_cells": moving_section.n_obs,
                       "reference_covered": int(valid[0].sum()), "moving_covered": int(valid[1].sum()),
                       "reference_valid_grids": valid_grids[0], "moving_valid_grids": valid_grids[1],
                       "native_insufficient_grid_fallback": bool(min(valid_grids) < 2)}
            assert [row["gene"] for row in gene_records] == config["evaluation_genes"]
            pair_rows.append({"Dataset": dataset, "pair_index": pair_index, **values, **support})
            gene_rows.extend({"pair_index": pair_index, "gene_argument_index": index, **row} for index, row in enumerate(gene_records))
            pd.DataFrame(pair_rows).to_csv(target / "pair_metrics.csv", index=False)
            save_json(OUTPUT_ROOT / "PROGRESS.json", {"dataset": dataset, "pairs_done": pair_index + 1, "pairs_expected": config["pairs"]})
            reference_slice = moving_slice
    assert len(pair_rows) == config["pairs"]
    means = {}
    for (display_name, native_key) in zip(METRICS, NATIVE_KEYS):
        numbers = [row[native_key] for row in pair_rows]
        assert len(numbers) == config["pairs"]
        means[display_name] = math.fsum(numbers) / len(numbers) if all(v is not None and math.isfinite(v) for v in numbers) else None
    pd.DataFrame(gene_rows).to_csv(target / "gene_metrics.csv.gz", index=False, compression="gzip")
    save_json(target / "RECOMPUTED.json", {"dataset": dataset, "pairs": config["pairs"], "metrics": means,
                                          "coordinates_sha256": config["coordinates_sha256"], "source_hashes_checked": VERIFY_SOURCE_HASHES})
    recomputed_rows.append({"Dataset": dataset, **means})
    print(f"{dataset}: {config['pairs']}/{config['pairs']} pairs recomputed")
recomputed = pd.DataFrame(recomputed_rows).set_index("Dataset")
recomputed.to_csv(OUTPUT_ROOT / "OmniAligner_NCS_recomputed.csv")
display(recomputed[METRICS].round(6))

## 5. Compare with the current result table
Each metric is checked separately at absolute tolerance `1e-10`, with no relative tolerance. A mismatch raises an error and leaves the recomputed files available for investigation; it never overwrites the reference table.

In [ ]:
expected = reference.loc[DATASETS, METRICS].astype(float)
actual = recomputed.loc[DATASETS, METRICS].astype(float)
difference = actual - expected
matches = np.isclose(actual.to_numpy(), expected.to_numpy(), atol=ATOL, rtol=0, equal_nan=False)
display(difference.rename_axis("Dataset: recomputed minus reference"))
validation = {"all_metrics_match": bool(matches.all()), "datasets": DATASETS,
              "metrics_checked": int(matches.size), "absolute_tolerance": ATOL,
              "maximum_absolute_difference": float(np.abs(difference.to_numpy()).max()),
              "elapsed_seconds": time.time() - started,
              "source_hashes_checked": VERIFY_SOURCE_HASHES,
              "packages": {name: importlib.metadata.version(name) for name in ["numpy", "scipy", "pandas", "anndata", "scanpy", "scikit-learn"]}}
save_json(OUTPUT_ROOT / "VALIDATION.json", validation)
assert matches.all(), "Recomputed metrics differ: inspect pair_metrics.csv and gene_metrics.csv.gz; the reference was not changed."
print(f"PASS: {matches.size}/{matches.size} metrics match the reference at absolute tolerance {ATOL:g}.")

## Interpretation
Successful agreement verifies reproduction of these selected saved results under their original scoring protocol. It does not establish independent anatomical validation, blind model selection, or equivalence between annotation consistency and biological fidelity. The weights, coordinates, source inputs, gene panels and cohort definitions must travel together when reproducing the results.